# GPU, node and SLURM log data integration and preprocessing 

This notebooks shows the processes from data filtering and preprocessing of 243 jobs training inception4 and EDA. 

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shutil

# Paths 

In [2]:
# This path points to the root directory where the data was extracted
ROOT_PATH = '/project/scratch/p200631/Silvana/datacenter-challenge/202201'

# The paths below point to specific files or directories
SCHEDULER_LOG_PATH = os.path.join(ROOT_PATH,'slurm-log.csv') # slurm log csv
NODE_DATA_PATH = os.path.join(ROOT_PATH,'node-data.csv') # node data csv
CPU_DATA_PATH = os.path.join(ROOT_PATH,'cpu') # cpu time series directory
GPU_DATA_PATH = os.path.join(ROOT_PATH,'gpu') # gpu time series directory

In [3]:
MAPPING_CSV= os.path.join(ROOT_PATH,'labelled_jobids.csv') # slurm log csv

## Filter the dataset to include only jobs which are for inception v4

In [5]:
df = pd.read_csv(MAPPING_CSV, dtype={'id_job': str, 'model': str})
inception4_jobs = set(df[df['model'].str.lower() == 'inception4']['id_job'])

print(f"Found {len(inception4_jobs)} Inception4 job IDs.")

Found 243 Inception4 job IDs.


In [30]:
DEST_FOLDER = '/project/scratch/p200631/Silvana/gpu_utilization/inception4_gpu_logs' 

In [9]:
os.makedirs(DEST_FOLDER, exist_ok=True)

In [ ]:
matched_count = 0

for root, dirs, files in os.walk(GPU_DATA_PATH):
    for fname in files:
        if fname.endswith('.csv'):
            job_id = fname.split('-')[0]
            if job_id in inception4_jobs:
                src_path = os.path.join(root, fname)
                dst_path = os.path.join(DEST_FOLDER, fname)
                shutil.copy2(src_path, dst_path)
                matched_count += 1

print(f"✅ Copied {matched_count} Inception4 GPU CSV files to: {DEST_FOLDER}")

✅ Copied 331 Inception4 GPU CSV files to: /project/scratch/p200631/Silvana/gpu_utilization/inception4_gpu_logs

### merge gpu logs with slurm logs based on job id 

In [13]:
slurm_df = pd.read_csv(SCHEDULER_LOG_PATH, dtype={'id_job': str})

In [14]:
slurm_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395914 entries, 0 to 395913
Data columns (total 29 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   id_job              395914 non-null  object 
 1   id_array_job        395914 non-null  int64  
 2   id_array_task       395914 non-null  int64  
 3   id_user             395914 non-null  int64  
 4   kill_requid         395914 non-null  int64  
 5   nodes_alloc         395914 non-null  int64  
 6   nodelist            395914 non-null  object 
 7   cpus_req            395914 non-null  int64  
 8   derived_ec          395914 non-null  int64  
 9   exit_code           395914 non-null  int64  
 10  gres_used           0 non-null       float64
 11  array_max_tasks     395914 non-null  int64  
 12  array_task_pending  395914 non-null  int64  
 13  constraints         395914 non-null  object 
 14  flags               395914 non-null  int64  
 15  mem_req             395914 non-nul

In [15]:
slurm_df.set_index('id_job', inplace=True)

In [5]:
OUTPUT_DIR = '/project/scratch/p200631/Silvana/gpu_utilization/inception4_gpu_slurm' 

In [6]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [31]:
INCEPTION_DATA_PATH = DEST_FOLDER

In [18]:
# Count only files
file_count = sum(
    1 for entry in os.listdir(OUTPUT_DIR)
    if os.path.isfile(os.path.join(OUTPUT_DIR, entry))
)

print(f"Number of files in '{OUTPUT_DIR}': {file_count}")

Number of files in '/project/scratch/p200631/Silvana/gpu_utilization/inception4_gpu_slurm': 329


In [19]:
merged_count = 329

for fname in os.listdir(INCEPTION_DATA_PATH):
    if not fname.endswith('.csv'):
        continue
        
    out_path = os.path.join(OUTPUT_DIR, fname)

    if os.path.exists(out_path):
        print(f"⏭️ File already exists in destination: {fname}")
        continue

    try:
        job_id = fname.split('-')[0]

        gpu_path = os.path.join(INCEPTION_DATA_PATH, fname)
        gpu_df = pd.read_csv(gpu_path)

        # Add job_id explicitly to GPU data
        gpu_df['id_job'] = job_id

        # Merge with Slurm data
        if job_id in slurm_df.index:
            slurm_row = slurm_df.loc[[job_id]]  # keep as DataFrame
            merged_df = gpu_df.merge(slurm_row.reset_index(), on='id_job', how='left')

            # Save merged file
            out_path = os.path.join(OUTPUT_DIR, fname)
            merged_df.to_csv(out_path, index=False)
            merged_count += 1
        else:
            print(f"⚠️ Job ID {job_id} not found in Slurm metadata. Skipping.")

    except Exception as e:
        print(f"❌ Error processing {fname}: {e}")

print(f"\n✅ Done! Merged {merged_count} GPU CSV files with Slurm metadata.")

⏭️ File already exists in destination: 45293811679196-r8333645-n851693.csv
⏭️ File already exists in destination: 54233701240846-r629115-n976057.csv
⏭️ File already exists in destination: 1173416277102-r8937440-n830961.csv
⏭️ File already exists in destination: 54599849969130-r8333645-n976057.csv
⏭️ File already exists in destination: 62504647031457-r3226521-n911952.csv
⏭️ File already exists in destination: 531177744168-r629115-n976057.csv
⏭️ File already exists in destination: 33309805818215-r4822976-n139058.csv
⏭️ File already exists in destination: 29358576358138-r1457839-n851693.csv
⏭️ File already exists in destination: 9729018528767-r5573787-n386398.csv
⏭️ File already exists in destination: 47901710600990-r7343737-n830961.csv
⏭️ File already exists in destination: 5650105580220-r7343737-n976057.csv
⏭️ File already exists in destination: 5275403476309-r9720335-n386398.csv
⏭️ File already exists in destination: 82397825020533-r2582019-n911952.csv
⏭️ File already exists in destina

In [17]:
#check if there are any corrupted files 
mismatch_count = 0
missing_dest = 0
checked_files = 0

for fname in os.listdir(INCEPTION_DATA_PATH):
    if not fname.endswith('.csv'):
        continue

    src_path = os.path.join(INCEPTION_DATA_PATH, fname)
    dest_path = os.path.join(OUTPUT_DIR, fname)

    if not os.path.exists(dest_path):
        print(f"❌ Destination file missing: {fname}")
        missing_dest += 1
        continue

    try:
        src_rows = pd.read_csv(src_path).shape[0]
        dest_rows = pd.read_csv(dest_path).shape[0]

        if src_rows != dest_rows:
            print(f"⚠️ Row mismatch in {fname}: source={src_rows}, destination={dest_rows}")
            mismatch_count += 1
        else:
            checked_files += 1

    except Exception as e:
        print(f"❌ Error reading {fname}: {e}")

# ------------------------------
# Summary
# ------------------------------
print(f"\n✅ Checked {checked_files} files.")
print(f"❗ {mismatch_count} files had row count mismatches.")
print(f"❗ {missing_dest} destination files were missing.")

⚠️ Row mismatch in 33352820358490-r4179716-n976057.csv: source=792218, destination=485925
⚠️ Row mismatch in 3419101415399-r3741709-n685852.csv: source=340710, destination=25167

✅ Checked 329 files.
❗ 2 files had row count mismatches.
❗ 0 destination files were missing.


In [21]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 340710 entries, 0 to 340709
Data columns (total 39 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   timestamp                340710 non-null  float64
 1   gpu_index                340710 non-null  int64  
 2   utilization_gpu_pct      340710 non-null  int64  
 3   utilization_memory_pct   340710 non-null  int64  
 4   memory_free_MiB          340710 non-null  int64  
 5   memory_used_MiB          340710 non-null  int64  
 6   temperature_gpu          340710 non-null  int64  
 7   temperature_memory       340710 non-null  int64  
 8   power_draw_W             340710 non-null  float64
 9   pcie_link_width_current  340710 non-null  int64  
 10  id_job                   340710 non-null  object 
 11  id_array_job             340710 non-null  int64  
 12  id_array_task            340710 non-null  int64  
 13  id_user                  340710 non-null  int64  
 14  kill

### Remove unnecessary columns for easier processing 

In [5]:
inception_gpu = pd.read_csv('inception4_gpu_slurm/70053314562-r9352821-n43543.csv')

In [8]:
missing_values = inception_gpu.isnull().sum()
print("\nMissing values in each column:")
print(missing_values)


Missing values in each column:
timestamp                       0
gpu_index                       0
utilization_gpu_pct             0
utilization_memory_pct          0
memory_free_MiB                 0
memory_used_MiB                 0
temperature_gpu                 0
temperature_memory              0
power_draw_W                    0
pcie_link_width_current         0
id_job                          0
id_array_job                    0
id_array_task                   0
id_user                         0
kill_requid                     0
nodes_alloc                     0
nodelist                        0
cpus_req                        0
derived_ec                      0
exit_code                       0
gres_used                  377112
array_max_tasks                 0
array_task_pending              0
constraints                     0
flags                           0
mem_req                         0
partition                       0
priority                        0
state           

In [21]:
INPUT_FOLDER = 'inception4_gpu_slurm/'     # Folder with merged CSVs
OUTPUT_FOLDER = 'cleaned_gpu_slurm/'  

In [22]:
# Create output directory if needed
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

In [23]:
columns_to_drop = {
    
    # === SLURM ADMINISTRATIVE (Drop) ===
    'id_array_job': 'Array job management - not energy relevant',
    'id_array_task': 'Task management - not needed',
    'id_user': 'User ID - not relevant for energy patterns',
    'kill_requid': 'Job termination info - not needed',
    'nodelist': 'Redundant with Node column',
    
    # === JOB STATUS/METADATA (Drop) ===
    'derived_ec': 'Exit code - not energy relevant',
    'exit_code': 'Job completion status - not needed',
    'state': 'All completed jobs, not informative',
    'priority': 'Scheduling priority - not energy relevant', 
    'partition': 'Queue info - likely same for all Inception-4',
    'flags': 'SLURM flags - not energy relevant',
    'track_steps': 'Step tracking - not needed',
    
    # === RESOURCE ALLOCATION (Partially Drop) ===
    'tres_alloc': 'Detailed resource allocation - redundant with gres_used',
    'tres_req': 'Resource requests - covered by cpus_req, mem_req',
    'array_max_tasks': 'Array job config - not relevant',
    'array_task_pending': 'Task management - not needed',
    'constraints': 'Hardware constraints - might be all same',
    'cpus_req': 'no need to deal with cpus',
    
    # === GPU TECHNICAL DETAILS (Drop) ===
    'pcie_link_width_current': 'PCIe bandwidth - rarely changes',
    
    # === NODE-LEVEL (Partially Drop) ===
    'UserPIDCount': 'Process count - not directly energy relevant',
    'LustreRPCTotals': 'Lustre filesystem calls - too detailed'
}

In [24]:
drop_keys = list(columns_to_drop.keys())

In [25]:
# -------------------------------
# File processing loop
# -------------------------------
for fname in os.listdir(INPUT_FOLDER):
    if not fname.endswith('.csv'):
        continue

    input_path = os.path.join(INPUT_FOLDER, fname)
    output_path = os.path.join(OUTPUT_FOLDER, fname)

    try:
        df = pd.read_csv(input_path)

        # Drop only columns that exist
        df = df.drop(columns=[col for col in drop_keys if col in df.columns])

        # Save cleaned file
        df.to_csv(output_path, index=False)
        print(f"✅ Cleaned: {fname}")

    except Exception as e:
        print(f"❌ Error processing {fname}: {e}")


✅ Cleaned: 54233701240846-r629115-n976057.csv
✅ Cleaned: 17825795509658-r9555635-n139058.csv
✅ Cleaned: 31810002298333-r3117156-n136082.csv
✅ Cleaned: 50025213645758-r8333645-n830961.csv
✅ Cleaned: 30927299534289-r9175025-n976057.csv
✅ Cleaned: 20837315505850-r3475376-n136082.csv
✅ Cleaned: 72356763042544-r9535192-n911952.csv
✅ Cleaned: 50071244191685-r4229531-n386398.csv
✅ Cleaned: 49355659834404-r9720335-n911952.csv
✅ Cleaned: 73863839880643-r9102715-n43543.csv
✅ Cleaned: 60593168283883-r8333645-n911952.csv
✅ Cleaned: 59428617181294-r3741709-n851693.csv
✅ Cleaned: 38984646323185-r2652301-n911952.csv
✅ Cleaned: 57956674462928-r7343737-n911952.csv
✅ Cleaned: 53863407622556-r629115-n976057.csv
✅ Cleaned: 90993771451923-r3741709-n976057.csv
✅ Cleaned: 53606254826829-r9102715-n851693.csv
✅ Cleaned: 55640367274726-r8937440-n911952.csv
✅ Cleaned: 45855550040191-r8062914-n139058.csv
✅ Cleaned: 58887925703090-r3041626-n685852.csv
✅ Cleaned: 73093598781980-r8642123-n911952.csv
✅ Cleaned: 54233

In [26]:
# Count only files
file_count = sum(
    1 for entry in os.listdir(OUTPUT_FOLDER)
    if os.path.isfile(os.path.join(OUTPUT_FOLDER, entry))
)

print(f"Number of files in '{OUTPUT_FOLDER}': {file_count}")

Number of files in 'cleaned_gpu_slurm/': 331


In [27]:
df.head()

,timestamp,gpu_index,utilization_gpu_pct,utilization_memory_pct,memory_free_MiB,memory_used_MiB,temperature_gpu,temperature_memory,power_draw_W,id_job,nodes_alloc,gres_used,mem_req,timelimit,time_submit,time_eligible,time_start,time_end,time_suspended,job_type
0,1.626678e+09,0,100,6,32510,0,62,62,56.06,36440233808457,2,NaN,9223372036854784308,4294967295,1626624022,1626624022,1626691911,1626710020,0,OTHER
1,1.626678e+09,1,100,6,32510,0,55,55,51.98,36440233808457,2,NaN,9223372036854784308,4294967295,1626624022,1626624022,1626691911,1626710020,0,OTHER
2,1.626678e+09,0,100,6,32510,0,62,62,56.10,36440233808457,2,NaN,9223372036854784308,4294967295,1626624022,1626624022,1626691911,1626710020,0,OTHER
3,1.626678e+09,1,100,6,32510,0,55,55,51.98,36440233808457,2,NaN,9223372036854784308,4294967295,1626624022,1626624022,1626691911,1626710020,0,OTHER
4,1.626678e+09,0,100,6,32510,0,61,62,56.02,36440233808457,2,NaN,9223372036854784308,4294967295,1626624022,1626624022,1626691911,1626710020,0,OTHER


In [15]:
from ydata_profiling import ProfileReport

In [29]:
profile = ProfileReport(df, title="Profiling Report")

In [31]:
profile

In [32]:
#after profiling it is understood that even these other columns are not relevant, so they are also dropped.
extra_columns_to_drop = {
    'time_submit': 'Submission time - not affecting energy consumption',
    'time_eligible': 'Queue time - not energy relevant',
    'time_suspended': 'Suspension events - likely empty',
    'job_type':'irrelevant',
    'gres_used': 'missing',
    'nodes_alloc': 'present in the file name',
    
}

In [33]:
drop_keys = list(extra_columns_to_drop.keys())

In [34]:
INPUT_FOLDER = OUTPUT_FOLDER

In [35]:
# -------------------------------
# File processing loop
# -------------------------------
for fname in os.listdir(INPUT_FOLDER):
    if not fname.endswith('.csv'):
        continue

    input_path = os.path.join(INPUT_FOLDER, fname)
    output_path = os.path.join(OUTPUT_FOLDER, fname)

    try:
        df = pd.read_csv(input_path)

        # Drop only columns that exist
        df = df.drop(columns=[col for col in drop_keys if col in df.columns])

        # Save cleaned file
        df.to_csv(output_path, index=False)
        print(f"✅ Cleaned: {fname}")

    except Exception as e:
        print(f"❌ Error processing {fname}: {e}")


✅ Cleaned: 74176342429884-r8937440-n43543.csv
✅ Cleaned: 61330628283480-r4229531-n911952.csv
✅ Cleaned: 53512654935447-r8333645-n851693.csv
✅ Cleaned: 33352820358490-r4179716-n976057.csv
✅ Cleaned: 4245408558597-r5130449-n208530.csv
✅ Cleaned: 11701784246856-r9175025-n911952.csv
✅ Cleaned: 41680801223685-r2998125-n208530.csv
✅ Cleaned: 80384695883841-r6760045-n830961.csv
✅ Cleaned: 16008623061291-r3405251-n208530.csv
✅ Cleaned: 24636145338445-r4822976-n139058.csv
✅ Cleaned: 18308236073276-r3405251-n208530.csv
✅ Cleaned: 9363816195518-r5189505-n911952.csv
✅ Cleaned: 16243287844930-r629115-n911952.csv
✅ Cleaned: 31091963887103-r5189505-n386398.csv
✅ Cleaned: 29770494582122-r8937440-n830961.csv
✅ Cleaned: 12963784416485-r4179716-n851693.csv
✅ Cleaned: 37611634321396-r3041626-n851693.csv
✅ Cleaned: 11887426389368-r9192091-n830961.csv
✅ Cleaned: 21624292191664-r5189505-n43543.csv
✅ Cleaned: 16008623061291-r7217787-n43543.csv
✅ Cleaned: 80873534651966-r9352821-n911952.csv
✅ Cleaned: 18512102

In [4]:
#UserPIDCount and LustreRPCTotals are columns that are not necessary to be used in processing and they are also dropped.

In [11]:
node_df = pd.read_csv('inception4_node_data.csv')

In [12]:
node_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7195980 entries, 0 to 7195979
Data columns (total 7 columns):
 #   Column                Dtype  
---  ------                -----  
 0   Node                  object 
 1   Time                  object 
 2   UserPIDCount          object 
 3   FSlatency             float64
 4   LoadAvg               float64
 5   MemoryFreeInactiveKB  float64
 6   LustreRPCTotals       float64
dtypes: float64(4), object(3)
memory usage: 384.3+ MB


In [13]:
node_columns_to_drop = {
    'UserPIDCount': 'too many missing values and no analysis will be done in user level',
    'LustreRPCTotals': 'not relevant'}

In [14]:
drop_keys = list(node_columns_to_drop.keys())

In [15]:
node_df = node_df.drop(columns=[col for col in drop_keys if col in node_df.columns])


In [16]:
node_df.to_csv('inception4_node_data_cleaned.csv', index=False)
print(f"✅ Cleaned node data")

✅ Cleaned node data


### Check granularity of node data

In [26]:
node_df = pd.read_csv(NODE_DATA_PATH)

In [17]:
node_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34655892 entries, 0 to 34655891
Data columns (total 7 columns):
 #   Column                Dtype  
---  ------                -----  
 0   Node                  object 
 1   Time                  float64
 2   UserPIDCount          object 
 3   FSlatency             float64
 4   LoadAvg               float64
 5   MemoryFreeInactiveKB  float64
 6   LustreRPCTotals       float64
dtypes: float64(5), object(2)
memory usage: 1.8+ GB


In [18]:
# 1. Sort by Node and Time so differences are meaningful
node_df = node_df.sort_values(['Node', 'Time'])

In [19]:
# 2. Calculate time differences per node using groupby and diff()
node_df['TimeDiff'] = node_df.groupby('Node')['Time'].diff()

In [20]:
node_df

,Node,Time,UserPIDCount,FSlatency,LoadAvg,MemoryFreeInactiveKB,LustreRPCTotals,TimeDiff
488222,r1018283-n146651,1.615285e+09,NaN,0.0,1.12,191.0,222.0,NaN
488523,r1018283-n146651,1.615285e+09,NaN,0.0,0.83,190.0,263.0,300.0
488824,r1018283-n146651,1.615286e+09,NaN,0.0,1.07,188.0,223.0,300.0
489125,r1018283-n146651,1.615286e+09,NaN,0.0,0.90,186.0,221.0,300.0
489426,r1018283-n146651,1.615286e+09,NaN,0.0,1.29,185.0,221.0,300.0
...,...,...,...,...,...,...,...,...
34652954,r9931077-n48252,1.633045e+09,NaN,0.0,0.00,126.0,221.0,300.0
34653682,r9931077-n48252,1.633045e+09,NaN,0.0,0.07,126.0,221.0,300.0
34654410,r9931077-n48252,1.633045e+09,NaN,0.0,0.00,126.0,221.0,300.0
34655138,r9931077-n48252,1.633046e+09,NaN,0.0,0.00,126.0,221.0,300.0


In [21]:
# 3. Drop the NaN values in TimeDiff (first entry per node will be NaN)
time_diffs = node_df.dropna(subset=['TimeDiff'])

In [22]:
# 4. Now for each node, find the most common granularity (mode of TimeDiff)
granularity_per_node = time_diffs.groupby('Node')['TimeDiff'].agg(lambda x: x.mode()[0])

In [23]:
# 5. (Optional) Check if the granularity is consistent within each node
# We can calculate the number of unique TimeDiff values per node
unique_diffs_count = time_diffs.groupby('Node')['TimeDiff'].nunique()

# 6. Combine results for review
result = pd.DataFrame({
    'Granularity': granularity_per_node,
    'UniqueGranularityCount': unique_diffs_count
})

# Display the first few rows
print(result.head())

# 7. If you want to check if all nodes have the same granularity, check unique values
print("\nUnique granularities across nodes:")
print(result['Granularity'].unique())


                  Granularity  UniqueGranularityCount
Node                                                 
r1018283-n146651        300.0                     203
r1018283-n181711        300.0                     170
r1018283-n325382        300.0                     189
r1018283-n392209        300.0                     172
r1018283-n468303        300.0                     177

Unique granularities across nodes:
[300.   0.]


### Filter the nodes which are used to run Inception4 from node data

In [27]:
node_df['Time'] = pd.to_datetime(node_df['Time'], unit='s')

In [28]:
node_df

,Node,Time,UserPIDCount,FSlatency,LoadAvg,MemoryFreeInactiveKB,LustreRPCTotals
0,r7217787-n911952,2021-02-28 23:55:01,91472915699408:11|1706828023724:15|65855960046...,0.0,29.24,371.0,26359.0
1,r4858666-n911952,2021-03-01 00:00:01,66720169194922:40|,0.0,4.07,363.0,228.0
2,r2582019-n911952,2021-02-28 23:55:01,22654259079669:47|,0.0,40.01,377.0,2006.0
3,r9040233-n911952,2021-02-28 23:55:01,91472915699408:7|12886809117418:29|53679664603...,0.0,29.58,369.0,8289.0
4,r4229531-n911952,2021-02-28 23:55:01,15914930715133:5|,0.0,0.26,390.0,227.0
...,...,...,...,...,...,...,...
34655887,r5970292-n48252,2021-09-30 23:50:01,NaN,0.0,0.00,126.0,221.0
34655888,r5132788-n48252,2021-09-30 23:50:01,NaN,0.0,0.00,127.0,221.0
34655889,r2522469-n48252,2021-09-30 23:50:02,7:2|,0.0,0.00,128.0,259.0
34655890,r1750190-n48252,2021-09-30 23:50:01,7:2|,0.0,0.00,129.0,229.0


In [32]:
files = os.listdir(INCEPTION_DATA_PATH)

In [33]:
node_names = []
for f in files:
    if '-' in f:
        node_name = f.split('-', 1)[1]  # everything after the first dash
        node_name = os.path.splitext(node_name)[0]  # remove file extension like '.csv'
        node_names.append(node_name)

In [34]:
len(node_names)

331

In [35]:
node_names_set = set(node_names)  # unique node names

print(f"Found {len(node_names_set)} unique nodes from GPU files.")

Found 147 unique nodes from GPU files.


There are 331 jobs which are run on a gpu for training inceptionv4 on 147 different nodes. 

In [36]:
# --- Step 4: Filter the big DataFrame for these nodes ---
filtered_df = node_df[node_df['Node'].isin(node_names_set)]

print(f"Filtered data contains {len(filtered_df)} rows.")

Filtered data contains 7195980 rows.


In [37]:
# Save filtered data to CSV ---
output_file = 'inception4_node_data.csv'
filtered_df.to_csv(output_file, index=False)

print(f"Filtered data saved to {output_file}")

Filtered data saved to inception4_node_data.csv


### Resample node data to 10s granularity 

In [38]:
# Step 1: Load the filtered node data (with inception4 nodes)
file_path = 'inception4_node_data.csv'
df = pd.read_csv(file_path)

In [42]:
df.head()

,Node,Time,UserPIDCount,FSlatency,LoadAvg,MemoryFreeInactiveKB,LustreRPCTotals
0,r7217787-n911952,2021-02-28 23:55:01,91472915699408:11|1706828023724:15|65855960046...,0.0,29.24,371.0,26359.0
1,r4858666-n911952,2021-03-01 00:00:01,66720169194922:40|,0.0,4.07,363.0,228.0
2,r2582019-n911952,2021-02-28 23:55:01,22654259079669:47|,0.0,40.01,377.0,2006.0
3,r9040233-n911952,2021-02-28 23:55:01,91472915699408:7|12886809117418:29|53679664603...,0.0,29.58,369.0,8289.0
4,r4229531-n911952,2021-02-28 23:55:01,15914930715133:5|,0.0,0.26,390.0,227.0


In [43]:
df['Time'] = pd.to_datetime(df['Time'], errors='coerce')

In [44]:
print(df['Time'].dtype)          # Should show: datetime64[ns]
print(df['Time'].head())         # Should show actual datetime objects

datetime64[ns]
0   2021-02-28 23:55:01
1   2021-03-01 00:00:01
2   2021-02-28 23:55:01
3   2021-02-28 23:55:01
4   2021-02-28 23:55:01
Name: Time, dtype: datetime64[ns]


In [45]:
# Step 2: How many unique nodes?
num_nodes = df['Node'].nunique()
print(f"Number of unique nodes: {num_nodes}")

Number of unique nodes: 147


In [46]:
# Step 3: Calculate granularity (most common time diff) per node
df = df.sort_values(['Node', 'Time'])
df['TimeDiff'] = df.groupby('Node')['Time'].diff()

# Most common granularity per node
granularity_per_node = df.groupby('Node')['TimeDiff'].agg(lambda x: x.mode()[0] if not x.mode().empty else pd.NaT)

print("\nGranularity per node (most common time diff):")
print(granularity_per_node)


Granularity per node (most common time diff):
Node
r1457839-n851693   0 days 00:05:00
r1485405-n685852   0 days 00:05:00
r1485405-n851693   0 days 00:05:00
r1485405-n911952   0 days 00:05:00
r1485405-n976057   0 days 00:05:00
                         ...      
r9535192-n976057   0 days 00:05:00
r9555635-n139058   0 days 00:05:00
r9555635-n208530   0 days 00:05:00
r9720335-n386398   0 days 00:05:00
r9720335-n911952   0 days 00:05:00
Name: TimeDiff, Length: 147, dtype: timedelta64[ns]


In [49]:
df.head()

,Node,Time,UserPIDCount,FSlatency,LoadAvg,MemoryFreeInactiveKB,LustreRPCTotals,TimeDiff
32,r1457839-n851693,2021-02-28 23:55:01,NaN,0.0,0.02,387.0,7245.0,NaT
179,r1457839-n851693,2021-03-01 00:05:01,NaN,0.0,0.00,387.0,220.0,0 days 00:10:00
326,r1457839-n851693,2021-03-01 00:10:01,NaN,0.0,0.03,387.0,220.0,0 days 00:05:00
473,r1457839-n851693,2021-03-01 00:15:01,NaN,0.0,0.00,387.0,220.0,0 days 00:05:00
620,r1457839-n851693,2021-03-01 00:20:01,NaN,0.0,0.02,387.0,220.0,0 days 00:05:00


### Check for duplicates 

In [57]:
len(df)

7195980

In [56]:
dupe_counts = (
    df
    .groupby(['Node', 'Time'])
    .size()
    .reset_index(name='Count')
    .query('Count > 1')
    .sort_values('Count', ascending=False)
)

print(f"Found {len(dupe_counts)} duplicate (Node, Time) pairs.")
print(dupe_counts.head())


Found 200048 duplicate (Node, Time) pairs.
                     Node                Time  Count
232572    r1682297-n43543 2021-04-15 07:50:01  14225
187705   r1485405-n976057 2021-03-29 11:40:02   5689
432627   r2652301-n685852 2021-03-15 11:55:01   2889
5327754   r9102715-n43543 2021-01-24 01:15:01   2174
432626   r2652301-n685852 2021-03-02 06:05:01   1930


In [58]:
# Example: get one (Node, Time) pair with duplicates
example_node = dupe_counts.iloc[0]['Node']
example_time = dupe_counts.iloc[0]['Time']

# Now filter and inspect those rows
duplicate_rows = df[(df['Node'] == example_node) & (df['Time'] == example_time)]
print(duplicate_rows)


                    Node                Time UserPIDCount  FSlatency  LoadAvg  \
1471404  r1682297-n43543 2021-04-15 07:50:01         7:2|        0.0      0.0   
1471551  r1682297-n43543 2021-04-15 07:50:01         7:2|        0.0      0.0   
1471698  r1682297-n43543 2021-04-15 07:50:01         7:2|        0.0      0.0   
1471845  r1682297-n43543 2021-04-15 07:50:01         7:2|        0.0      0.0   
1471992  r1682297-n43543 2021-04-15 07:50:01         7:2|        0.0      0.0   
...                  ...                 ...          ...        ...      ...   
3561489  r1682297-n43543 2021-04-15 07:50:01         7:2|        0.0      0.0   
3561634  r1682297-n43543 2021-04-15 07:50:01         7:2|        0.0      0.0   
3561779  r1682297-n43543 2021-04-15 07:50:01         7:2|        0.0      0.0   
3561924  r1682297-n43543 2021-04-15 07:50:01         7:2|        0.0      0.0   
3562068  r1682297-n43543 2021-04-15 07:50:01         7:2|        0.0      0.0   

         MemoryFreeInactive

In [59]:
# Step 5: Resample each node's data to 10-second intervals
# We'll set 'Time' as index for resampling, then group by Node and resample

resampled_list = []

for node, group in df.groupby('Node'):
    group = group.set_index('Time').sort_index()

    group = group[~group.index.duplicated(keep='first')]
    
    # Resample at 10-second frequency, forward-fill missing values (adjust as needed)
    resampled_group = group.resample('10S').ffill().reset_index()
    
    # Add back the Node column
    resampled_group['Node'] = node
    
    resampled_list.append(resampled_group)

# Combine all nodes back into one DataFrame
df_resampled = pd.concat(resampled_list, ignore_index=True)

print(f"\nResampled data shape: {df_resampled.shape}")



Resampled data shape: (272971528, 8)


In [60]:
# Step 4: Check for missing values in the DataFrame
missing_values = df.isnull().sum()
print("\nMissing values in each column:")
print(missing_values)


Missing values in each column:
Node                         0
Time                         0
UserPIDCount            717881
FSlatency                    0
LoadAvg                      0
MemoryFreeInactiveKB         0
LustreRPCTotals              0
TimeDiff                   147
dtype: int64


In [ ]:
#  Save resampled data
df_resampled.to_csv('inception4_node_logs_resampled_10s.csv', index=False)
print("\nResampled data saved to 'inception4_node_logs_resampled_10s.csv'")


In [2]:
resampled = pd.read_csv('inception4_node_logs_resampled_10s.csv')

/tmp/ipykernel_1033/2653808312.py:1: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  resampled = pd.read_csv('inception4_node_logs_resampled_10s.csv')


In [3]:
resampled.shape

(31475000, 8)

### Clean duplicates from node data

In [26]:
def analyze_node_duplicates(node_df):
        """
        Analyze duplicate patterns in node data
        """
        
        print("=== NODE DATA DUPLICATE ANALYSIS ===")
        
        # Check for duplicates by (Node, Time)
        duplicates = node_df.groupby(['Node', 'Time']).size()
        duplicate_pairs = duplicates[duplicates > 1]
        
        print(f"Total node records: {len(node_df):,}")
        print(f"Unique (Node, timestamp) pairs: {len(duplicates):,}")
        print(f"Duplicate (Node, timestamp) pairs: {len(duplicate_pairs):,}")
        
        if len(duplicate_pairs) > 0:
            print(f"Max duplicates for single (Node, timestamp): {duplicate_pairs.max()}")
            
            # Show examples of duplicates
            print(f"\nSample duplicate records:")
            sample_duplicate = duplicate_pairs.head(3)
            
            for (node, timestamp), count in sample_duplicate.items():
                print(f"  Node {node}, {timestamp}: {count} records")
                
                # Show the actual duplicate records
                duplicate_records = node_df[
                    (node_df['Node'] == node) & 
                    (node_df['Time'] == timestamp)
                ]
                print(f"    Values: {duplicate_records[['LoadAvg', 'MemoryFreeInactiveKB']].to_dict('records')}")
        
        return duplicate_pairs

In [29]:
node_df

,Node,Time,FSlatency,LoadAvg,MemoryFreeInactiveKB
0,r7217787-n911952,2021-02-28 23:55:01,0.0,29.24,371.0
1,r4858666-n911952,2021-03-01 00:00:01,0.0,4.07,363.0
2,r2582019-n911952,2021-02-28 23:55:01,0.0,40.01,377.0
3,r9040233-n911952,2021-02-28 23:55:01,0.0,29.58,369.0
4,r4229531-n911952,2021-02-28 23:55:01,0.0,0.26,390.0
...,...,...,...,...,...
7195975,r5189505-n830961,2021-09-30 23:50:02,0.0,2.52,383.0
7195976,r9192091-n830961,2021-09-30 23:50:01,0.0,2.49,383.0
7195977,r3226521-n830961,2021-09-30 23:50:01,0.0,2.21,383.0
7195978,r8937440-n830961,2021-09-30 23:50:01,0.0,2.50,385.0


In [28]:
analyze_node_duplicates(node_df)

=== NODE DATA DUPLICATE ANALYSIS ===
Total node records: 7,195,980
Unique (Node, timestamp) pairs: 6,640,176
Duplicate (Node, timestamp) pairs: 200,048
Max duplicates for single (Node, timestamp): 14225

Sample duplicate records:
  Node r1457839-n851693, 2021-03-01 04:05:01: 2 records
    Values: [{'LoadAvg': 0.24, 'MemoryFreeInactiveKB': 387.0}, {'LoadAvg': 0.24, 'MemoryFreeInactiveKB': 387.0}]
  Node r1457839-n851693, 2021-03-01 04:55:01: 2 records
    Values: [{'LoadAvg': 0.37, 'MemoryFreeInactiveKB': 387.0}, {'LoadAvg': 0.37, 'MemoryFreeInactiveKB': 387.0}]
  Node r1457839-n851693, 2021-03-01 07:25:01: 2 records
    Values: [{'LoadAvg': 0.22, 'MemoryFreeInactiveKB': 387.0}, {'LoadAvg': 0.22, 'MemoryFreeInactiveKB': 387.0}]


Node              Time               
r1457839-n851693  2021-03-01 04:05:01    2
                  2021-03-01 04:55:01    2
                  2021-03-01 07:25:01    2
                  2021-03-02 18:40:01    2
                  2021-03-02 19:35:01    2
                                        ..
r9720335-n911952  2021-09-30 21:30:01    2
                  2021-09-30 22:20:01    2
                  2021-09-30 22:50:01    2
                  2021-09-30 23:20:01    2
                  2021-09-30 23:35:01    2
Length: 200048, dtype: int64

In [38]:
def clean_node_duplicates(node_df, strategy='mean'):
        """
        Clean duplicates in node data
        
        Parameters:
        - strategy: 'mean', 'first', 'last', 'median'
        """
        
        print(f"Cleaning node duplicates using strategy: {strategy}")
        
        original_size = len(node_df)
        
        # Group by Node and timestamp, then aggregate
        if strategy == 'mean':
            # Take mean of numeric columns for duplicates
            numeric_cols = node_df.select_dtypes(include=[np.number]).columns
            agg_dict = {col: 'mean' for col in numeric_cols if col != 'timestamp'}
            agg_dict.update({col: 'first' for col in node_df.columns if col not in numeric_cols and col != 'timestamp'})
            
        elif strategy == 'first':
            # Take first occurrence
            agg_dict = {col: 'first' for col in node_df.columns if col != 'timestamp'}
            
        elif strategy == 'last':
            # Take last occurrence  
            agg_dict = {col: 'last' for col in node_df.columns if col != 'timestamp'}
            
        elif strategy == 'median':
            # Take median of numeric columns
            numeric_cols = node_df.select_dtypes(include=[np.number]).columns
            agg_dict = {col: 'median' for col in numeric_cols if col != 'timestamp'}
            agg_dict.update({col: 'first' for col in node_df.columns if col not in numeric_cols and col != 'timestamp'})
        
        else:
            raise ValueError(f"Unknown strategy: {strategy}")
        
        # Remove timestamp from aggregation (it's in the groupby)
        if 'timestamp' in agg_dict:
            del agg_dict['timestamp']
        if 'Node' in agg_dict:
            del agg_dict['Node']
        
        # Perform aggregation
        cleaned_df = node_df.groupby(['Node', 'timestamp']).agg(agg_dict).reset_index()
        
        print(f"Original node records: {original_size:,}")
        print(f"Cleaned node records: {len(cleaned_df):,}")
        print(f"Duplicates removed: {original_size - len(cleaned_df):,}")
        
        return cleaned_df

In [46]:
node_df.rename(columns={'Time':'timestamp'}, inplace=True)

In [47]:
node_df

,Node,timestamp,FSlatency,LoadAvg,MemoryFreeInactiveKB
0,r7217787-n911952,2021-02-28 23:55:01,0.0,29.24,371.0
1,r4858666-n911952,2021-03-01 00:00:01,0.0,4.07,363.0
2,r2582019-n911952,2021-02-28 23:55:01,0.0,40.01,377.0
3,r9040233-n911952,2021-02-28 23:55:01,0.0,29.58,369.0
4,r4229531-n911952,2021-02-28 23:55:01,0.0,0.26,390.0
...,...,...,...,...,...
7195975,r5189505-n830961,2021-09-30 23:50:02,0.0,2.52,383.0
7195976,r9192091-n830961,2021-09-30 23:50:01,0.0,2.49,383.0
7195977,r3226521-n830961,2021-09-30 23:50:01,0.0,2.21,383.0
7195978,r8937440-n830961,2021-09-30 23:50:01,0.0,2.50,385.0


In [48]:
#Since the duplicates have exactly the same values for other columns, I will use 'first' strategy to clean the duplicates 
cleaned_node = clean_node_duplicates(node_df, strategy='first')

Cleaning node duplicates using strategy: first
Original node records: 7,195,980
Cleaned node records: 6,640,176
Duplicates removed: 555,804


In [49]:
cleaned_node.rename(columns={'timestamp':'Time'}, inplace=True)


In [51]:
analyze_node_duplicates(cleaned_node)

=== NODE DATA DUPLICATE ANALYSIS ===
Total node records: 6,640,176
Unique (Node, timestamp) pairs: 6,640,176
Duplicate (Node, timestamp) pairs: 0


Series([], dtype: int64)

In [79]:
cleaned_node.to_csv('inception4_node_no_duplicates.csv')

In [83]:
cleaned_node.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6640176 entries, 0 to 6640175
Data columns (total 5 columns):
 #   Column                Dtype         
---  ------                -----         
 0   Node                  object        
 1   timestamp             datetime64[ns]
 2   FSlatency             float64       
 3   LoadAvg               float64       
 4   MemoryFreeInactiveKB  float64       
dtypes: datetime64[ns](1), float64(3), object(1)
memory usage: 253.3+ MB


### Merge cleaned gpu data and node data

In [105]:
gpu_file_path = 'cleaned_gpu_slurm/70053314562-r9352821-n43543.csv'

In [106]:
filename = os.path.basename(gpu_file_path)
    
# Assuming format: jobid-nodename.csv or similar
if '-' in filename:
    node = filename.split('-', 1)[1].replace('.csv', '')
else:
    raise ValueError(f"Unexpected filename format: {filename}")

In [107]:
node

'r9352821-n43543'

In [108]:
gpu_df = pd.read_csv(gpu_file_path)

In [109]:
gpu_df

,timestamp,gpu_index,utilization_gpu_pct,utilization_memory_pct,memory_free_MiB,memory_used_MiB,temperature_gpu,temperature_memory,power_draw_W,id_job,mem_req,timelimit,time_start,time_end
0,1.623182e+09,0,0,0,32510,0,41,39,44.16,70053314562,9223372036854784308,4294967295,1623196040,1623235016
1,1.623182e+09,0,0,0,32510,0,41,39,44.26,70053314562,9223372036854784308,4294967295,1623196040,1623235016
2,1.623182e+09,0,0,0,32510,0,41,39,44.08,70053314562,9223372036854784308,4294967295,1623196040,1623235016
3,1.623182e+09,0,0,0,32510,0,41,39,44.16,70053314562,9223372036854784308,4294967295,1623196040,1623235016
4,1.623182e+09,0,0,0,32506,4,40,39,44.16,70053314562,9223372036854784308,4294967295,1623196040,1623235016
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
377107,1.623221e+09,0,0,0,32506,4,62,63,54.27,70053314562,9223372036854784308,4294967295,1623196040,1623235016
377108,1.623221e+09,0,54,3,32510,0,62,63,55.37,70053314562,9223372036854784308,4294967295,1623196040,1623235016
377109,1.623221e+09,0,54,3,32510,0,62,63,55.63,70053314562,9223372036854784308,4294967295,1623196040,1623235016
377110,1.623221e+09,0,100,6,32510,0,62,63,55.35,70053314562,9223372036854784308,4294967295,1623196040,1623235016


In [110]:
# Step 3: Convert timestamp
gpu_df['timestamp'] = pd.to_datetime(gpu_df['timestamp'], unit='s')

# Step 4: Add Node column
gpu_df['Node'] = node

In [111]:
# Sort cleaned_node by Node and timestamp (safe inside loop)
cleaned_node_sorted = cleaned_node.sort_values(['Node', 'timestamp'])

In [112]:
cleaned_node_1 = cleaned_node_sorted[cleaned_node_sorted.Node == 'r1457839-n851693']

In [113]:
cleaned_node_1

,Node,timestamp,FSlatency,LoadAvg,MemoryFreeInactiveKB
0,r1457839-n851693,2021-02-28 23:55:01,0.0,0.02,387.0
1,r1457839-n851693,2021-03-01 00:05:01,0.0,0.00,387.0
2,r1457839-n851693,2021-03-01 00:10:01,0.0,0.03,387.0
3,r1457839-n851693,2021-03-01 00:15:01,0.0,0.00,387.0
4,r1457839-n851693,2021-03-01 00:20:01,0.0,0.02,387.0
...,...,...,...,...,...
45567,r1457839-n851693,2021-09-30 23:25:01,0.0,3.63,271.0
45568,r1457839-n851693,2021-09-30 23:30:01,0.0,3.00,271.0
45569,r1457839-n851693,2021-09-30 23:35:01,0.0,3.49,271.0
45570,r1457839-n851693,2021-09-30 23:45:01,0.0,4.37,271.0


In [114]:
gpu_df_sorted = gpu_df.sort_values(['Node', 'timestamp'])

In [115]:
gpu_df_sorted

,timestamp,gpu_index,utilization_gpu_pct,utilization_memory_pct,memory_free_MiB,memory_used_MiB,temperature_gpu,temperature_memory,power_draw_W,id_job,mem_req,timelimit,time_start,time_end,Node
0,2021-06-08 19:47:20.099000064,0,0,0,32510,0,41,39,44.16,70053314562,9223372036854784308,4294967295,1623196040,1623235016,r9352821-n43543
1,2021-06-08 19:47:20.201999872,0,0,0,32510,0,41,39,44.26,70053314562,9223372036854784308,4294967295,1623196040,1623235016,r9352821-n43543
2,2021-06-08 19:47:20.305000192,0,0,0,32510,0,41,39,44.08,70053314562,9223372036854784308,4294967295,1623196040,1623235016,r9352821-n43543
3,2021-06-08 19:47:20.408999936,0,0,0,32510,0,41,39,44.16,70053314562,9223372036854784308,4294967295,1623196040,1623235016,r9352821-n43543
4,2021-06-08 19:47:20.513999872,0,0,0,32506,4,40,39,44.16,70053314562,9223372036854784308,4294967295,1623196040,1623235016,r9352821-n43543
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
377107,2021-06-09 06:36:56.599000064,0,0,0,32506,4,62,63,54.27,70053314562,9223372036854784308,4294967295,1623196040,1623235016,r9352821-n43543
377108,2021-06-09 06:36:56.749000192,0,54,3,32510,0,62,63,55.37,70053314562,9223372036854784308,4294967295,1623196040,1623235016,r9352821-n43543
377109,2021-06-09 06:36:56.852000000,0,54,3,32510,0,62,63,55.63,70053314562,9223372036854784308,4294967295,1623196040,1623235016,r9352821-n43543
377110,2021-06-09 06:36:56.954999808,0,100,6,32510,0,62,63,55.35,70053314562,9223372036854784308,4294967295,1623196040,1623235016,r9352821-n43543


In [123]:
node

'r9352821-n43543'

In [125]:
node_data = cleaned_node[cleaned_node['Node'] == node].copy()

if node_data.empty:
    print(f"⚠️ No matching node data for {node_name}, skipping")

In [121]:
# Try one Node only
node_name = cleaned_node_sorted['Node'].iloc[0]
print(node_name)
print("type of node name", type( node_name))
print("type of node in gpu df",type( gpu_df['Node'][0]))
if gpu_df['Node'].any() == node_name: print("found the node")
gpu_sub = gpu_df[gpu_df['Node'] == node_name].sort_values('timestamp')
node_sub = cleaned_node_sorted[cleaned_node_sorted['Node'] == node_name].sort_values('timestamp')

# Try merge
merged_df = pd.merge_asof(gpu_sub, node_sub, on='timestamp', direction='backward')


r1457839-n851693
type of node name <class 'str'>
type of node in gpu df <class 'str'>


In [4]:
from tqdm import tqdm
def load_and_merge_gpu_with_node_data(gpu_logs_folder, cleaned_node):
    all_merged = []

    for filename in tqdm(os.listdir(gpu_logs_folder)):
        if not filename.endswith('.csv'):
            continue

    
        # format: jobid-nodename.csv or similar
        if '-' in filename:
            node_name = filename.split('-', 1)[1].replace('.csv', '')
        else:
            raise ValueError(f"Unexpected filename format: {filename}")

        file_path = os.path.join(gpu_logs_folder, filename)
        
                
        # Load GPU log
        gpu_df = pd.read_csv(file_path)
        gpu_df['Node'] = node_name

        # Convert GPU timestamp (assumed to be float64 UNIX) to datetime
        gpu_df['timestamp'] = pd.to_datetime(gpu_df['timestamp'], unit='s')

        # Filter node data for this specific node
        node_data = cleaned_node[cleaned_node['Node'] == node_name].copy()

        if node_data.empty:
            print(f"⚠️ No matching node data for {node_name}, skipping")
            continue

        # Ensure timestamps are datetime
        if node_data['timestamp'].dtype == 'object':
            node_data['timestamp'] = pd.to_datetime(node_data['timestamp'])

        # Sort both by timestamp (and Node for asof)
        gpu_df_sorted = gpu_df.sort_values(by=["timestamp"])
        node_data_sorted = node_data.sort_values(by=["timestamp"])

        #forward fill node data and resample it to 10 seconds
        node_data_sorted = (
            node_data_sorted
            .set_index('timestamp')       # Use both as index                    
            .resample('10s')               # Resample on timestamp level
            .ffill()
            .reset_index()                          # Brings both timestamp back as column
        )

        # Merge asof with tolerance (e.g. 5 minutes)
        merged = pd.merge_asof(
            gpu_df_sorted,
            node_data_sorted,
            on="timestamp",
            direction="backward",
            tolerance=pd.Timedelta("2m"),
            suffixes=("", "_node")
        )

        all_merged.append(merged)

    # Combine all merged dataframes into one
    final_df = pd.concat(all_merged, ignore_index=True)
    return final_df


In [5]:
gpu_logs_folder='/project/scratch/p200631/Silvana/gpu_utilization/cleaned_gpu_slurm'

In [6]:
cleaned_node = pd.read_csv('inception4_node_no_duplicates.csv')

In [7]:
cleaned_node = cleaned_node.drop('Unnamed: 0', axis=1) 

In [8]:
cleaned_node

,Node,timestamp,FSlatency,LoadAvg,MemoryFreeInactiveKB
0,r1457839-n851693,2021-02-28 23:55:01,0.0,0.02,387.0
1,r1457839-n851693,2021-03-01 00:05:01,0.0,0.00,387.0
2,r1457839-n851693,2021-03-01 00:10:01,0.0,0.03,387.0
3,r1457839-n851693,2021-03-01 00:15:01,0.0,0.00,387.0
4,r1457839-n851693,2021-03-01 00:20:01,0.0,0.02,387.0
...,...,...,...,...,...
6640171,r9720335-n911952,2021-09-30 23:20:01,0.0,1.98,362.0
6640172,r9720335-n911952,2021-09-30 23:30:01,0.0,2.07,362.0
6640173,r9720335-n911952,2021-09-30 23:35:01,0.0,2.32,362.0
6640174,r9720335-n911952,2021-09-30 23:45:01,0.0,2.12,363.0


In [9]:
merged_df_resampled = load_and_merge_gpu_with_node_data(gpu_logs_folder, cleaned_node)

100%|██████████| 332/332 [04:39<00:00,  1.19it/s]


In [ ]:
merged_df_resampled.to_csv('merged_gpu_node_resampled_final.csv')

### Validate the merged dataframe

In [12]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 116796132 entries, 0 to 116796131
Data columns (total 24 columns):
 #   Column                     Dtype         
---  ------                     -----         
 0   timestamp                  datetime64[ns]
 1   gpu_index                  int64         
 2   utilization_gpu_pct        int64         
 3   utilization_memory_pct     int64         
 4   memory_free_MiB            int64         
 5   memory_used_MiB            int64         
 6   temperature_gpu            int64         
 7   temperature_memory         int64         
 8   power_draw_W               float64       
 9   id_job                     int64         
 10  mem_req                    uint64        
 11  timelimit                  int64         
 12  time_start                 int64         
 13  time_end                   int64         
 14  Node                       object        
 15  Unnamed: 0                 float64       
 16  Node_node                  objec

In [13]:
merged_df.isna().sum()


timestamp                           0
gpu_index                           0
utilization_gpu_pct                 0
utilization_memory_pct              0
memory_free_MiB                     0
memory_used_MiB                     0
temperature_gpu                     0
temperature_memory                  0
power_draw_W                        0
id_job                              0
mem_req                             0
timelimit                           0
time_start                          0
time_end                            0
Node                                0
Unnamed: 0                   68157858
Node_node                    68157858
FSlatency                    68157858
LoadAvg                      68157858
MemoryFreeInactiveKB         68157858
clocks_current_sm_MHz        89766748
clocks_current_memory_MHz    89766748
clocks_current_video_MHz     89766748
power_limit_W                89766748
dtype: int64

In [ ]:
profile = ProfileReport(merged_df, title="Profiling Report")

In [15]:
merged_df['Unnamed: 0']

0           NaN
1           NaN
2           NaN
3           NaN
4           NaN
             ..
116796127   NaN
116796128   NaN
116796129   NaN
116796130   NaN
116796131   NaN
Name: Unnamed: 0, Length: 116796132, dtype: float64

In [16]:
gpu_nodes = set(merged_df['Node'].unique())
node_nodes = set(cleaned_node['Node'].unique())
missing_nodes = gpu_nodes - node_nodes
print("Nodes in GPU logs but not in cleaned_node:", missing_nodes)


Nodes in GPU logs but not in cleaned_node: set()


#### Validation of the dataframe merged after resampling node data

In [11]:
merged_df_resampled.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 116796132 entries, 0 to 116796131
Data columns (total 23 columns):
 #   Column                     Dtype         
---  ------                     -----         
 0   timestamp                  datetime64[ns]
 1   gpu_index                  int64         
 2   utilization_gpu_pct        int64         
 3   utilization_memory_pct     int64         
 4   memory_free_MiB            int64         
 5   memory_used_MiB            int64         
 6   temperature_gpu            int64         
 7   temperature_memory         int64         
 8   power_draw_W               float64       
 9   id_job                     int64         
 10  mem_req                    uint64        
 11  timelimit                  int64         
 12  time_start                 int64         
 13  time_end                   int64         
 14  Node                       object        
 15  Node_node                  object        
 16  FSlatency                  float

In [12]:
merged_df_resampled.isna().sum()

timestamp                           0
gpu_index                           0
utilization_gpu_pct                 0
utilization_memory_pct              0
memory_free_MiB                     0
memory_used_MiB                     0
temperature_gpu                     0
temperature_memory                  0
power_draw_W                        0
id_job                              0
mem_req                             0
timelimit                           0
time_start                          0
time_end                            0
Node                                0
Node_node                     1174692
FSlatency                     1174692
LoadAvg                       1174692
MemoryFreeInactiveKB          1174692
clocks_current_sm_MHz        89766748
clocks_current_memory_MHz    89766748
clocks_current_video_MHz     89766748
power_limit_W                89766748
dtype: int64

In [18]:
ROOT_PATH

'/project/scratch/p200631/Silvana/datacenter-challenge/202201'

In [22]:
import os
import pandas as pd

gpu_folder = GPU_DATA_PATH
columns_present = []

for root, dirs, files in os.walk(GPU_DATA_PATH):
    for file in files:
        if file.endswith('.csv'):
            file_path = os.path.join(root, file)
            df = pd.read_csv(file_path, nrows=5)
            columns_present.append((file, df.columns.tolist()))

# Print files missing those columns
count=0
for file, cols in columns_present:
    missing = [col for col in ['clocks_current_sm_MHz', 'clocks_current_memory_MHz', 'clocks_current_video_MHz', 'power_limit_W'] if col not in cols]
    if missing:
        count+=1
print(f"{count} files have missing columns")


97402 files have missing columns


The missing data for clock and powerlimit comes from the source, so the information is not lost during any filtering and preprocessing phase. 

In [25]:
#### Remove unnecessary columns and save the output inplace 
merged_df_resampled.drop(['Node_node', 'clocks_current_video_MHz'], axis=1, inplace=True)

In [26]:
merged_df_resampled.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 116796132 entries, 0 to 116796131
Data columns (total 21 columns):
 #   Column                     Dtype         
---  ------                     -----         
 0   timestamp                  datetime64[ns]
 1   gpu_index                  int64         
 2   utilization_gpu_pct        int64         
 3   utilization_memory_pct     int64         
 4   memory_free_MiB            int64         
 5   memory_used_MiB            int64         
 6   temperature_gpu            int64         
 7   temperature_memory         int64         
 8   power_draw_W               float64       
 9   id_job                     int64         
 10  mem_req                    uint64        
 11  timelimit                  int64         
 12  time_start                 int64         
 13  time_end                   int64         
 14  Node                       object        
 15  FSlatency                  float64       
 16  LoadAvg                    float